# Medical Assistant - Part 4: LangChain Pipeline

## 1 - Setup

In [1]:
!pip install --quiet langchain langchain-community langchain-core langchain-text-splitters langgraph sentence-transformers faiss-cpu grandalf


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import json
import sqlite3
import logging
from datetime import datetime
from pathlib import Path
from typing import List, Optional, TypedDict

PROJECT_DIR = Path(__file__).parent if "__file__" in dir() else Path.cwd()
DATA_DIR    = PROJECT_DIR / "data"
OUTPUTS_DIR = PROJECT_DIR / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL    = "phi3"

# Audit logger
log_file = OUTPUTS_DIR / "medical_assistant.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger("MedicalAssistant")

print(f"✅ Project dir : {PROJECT_DIR}")
print(f"✅ Ollama URL  : {OLLAMA_BASE_URL}  model={OLLAMA_MODEL}")
print(f"✅ Log file    : {log_file}")

✅ Project dir : /Users/anabarbosa/fiap/fiap-tech-challenge/module_03
✅ Ollama URL  : http://localhost:11434  model=phi3
✅ Log file    : /Users/anabarbosa/fiap/fiap-tech-challenge/module_03/outputs/medical_assistant.log


## 2 - Connect to Ollama (LLM)

In [3]:
from langchain_community.llms import Ollama

llm = Ollama(
    base_url=OLLAMA_BASE_URL,
    model=OLLAMA_MODEL,
    temperature=0.3,
    num_predict=512,
)

# Verify connection
try:
    test = llm.invoke("In one sentence, what is hypertension?")
    print("✅ Ollama connected")
    print(f"   Test: {test.strip()}")
except Exception as e:
    print(f"❌ Cannot reach Ollama at {OLLAMA_BASE_URL}: {e}")
    print("   Make sure Ollama is running on your host machine.")

/var/folders/jg/dxyd05n13qb115cb0brztg6c0000gn/T/ipykernel_83604/31365876.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import Ollama
/Users/anabarbosa/fiap/fiap-tech-challenge/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/jg/dxyd05n13qb115cb0brztg6c0000gn/T/ipykernel_83604/31365876.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import a

✅ Ollama connected
   Test: Hypertension is a chronic medical condition in which the blood pressure in the arteries is persistently elevated.


## 3 - Synthetic Patient Database (SQLite)

In [4]:
DB_PATH = str(DATA_DIR / "patients.db")

# Clean up existing database
if Path(DB_PATH).exists():
    Path(DB_PATH).unlink()

def create_patient_database():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()

    c.executescript("""
        CREATE TABLE IF NOT EXISTS patients (
            id          INTEGER PRIMARY KEY,
            name        TEXT,
            age         INTEGER,
            gender      TEXT,
            diagnosis   TEXT,
            medications TEXT,
            allergies   TEXT,
            blood_type  TEXT,
            doctor      TEXT,
            last_visit  TEXT
        );
        CREATE TABLE IF NOT EXISTS exams (
            id           INTEGER PRIMARY KEY,
            patient_id   INTEGER,
            exam_type    TEXT,
            status       TEXT,
            ordered_date TEXT,
            result_date  TEXT,
            results      TEXT,
            FOREIGN KEY (patient_id) REFERENCES patients(id)
        );
        CREATE TABLE IF NOT EXISTS appointments (
            id               INTEGER PRIMARY KEY,
            patient_id       INTEGER,
            appointment_type TEXT,
            scheduled_date   TEXT,
            status           TEXT,
            notes            TEXT,
            FOREIGN KEY (patient_id) REFERENCES patients(id)
        );
    """)

    c.executemany("INSERT OR REPLACE INTO patients VALUES (?,?,?,?,?,?,?,?,?,?)", [
        (1, "João Silva"      , 65, "M", "Hypertension, Type 2 Diabetes"             , "Metformin 500mg, Lisinopril 10mg"             , "Penicillin" , "A+" , "Dr. Ana Costa"   , "2026-04-15"),
        (2, "Maria Santos"    , 45, "F", "Asthma, Allergic Rhinitis"                 , "Salbutamol inhaler, Fluticasone 250mcg"       , "NSAIDs"     , "O-" , "Dr. Pedro Lima"  , "2026-04-20"),
        (3, "Carlos Oliveira" , 72, "M", "Chronic Heart Failure, Atrial Fibrillation", "Furosemide 40mg, Warfarin 5mg, Bisoprolol 5mg", "Sulfa drugs", "B+" , "Dr. Ana Costa"   , "2026-05-01"),
        (4, "Ana Rodrigues"   , 38, "F", "Hypothyroidism, Depression"                , "Levothyroxine 100mcg, Sertraline 50mg"        , "None known" , "AB+", "Dr. Carla Mendes", "2026-05-03"),
        (5, "Roberto Ferreira", 55, "M", "Type 2 Diabetes, Dyslipidemia"             , "Metformin 1000mg, Atorvastatin 20mg"          , "Aspirin"    , "A-" , "Dr. Pedro Lima"  , "2026-04-28"),
    ])

    c.executemany("INSERT OR REPLACE INTO exams VALUES (?,?,?,?,?,?,?)", [
        (1, 1, "HbA1c"                        , "PENDING"  , "2026-05-01", None        , None                                                         ),
        (2, 1, "Renal Function Panel"         , "PENDING"  , "2026-05-01", None        , None                                                         ),
        (3, 1, "Lipid Panel"                  , "COMPLETED", "2026-04-15", "2026-04-18", "Total Cholesterol: 210 mg/dL, LDL: 130 mg/dL, HDL: 45 mg/dL"),
        (4, 2, "Spirometry"                   , "COMPLETED", "2026-04-20", "2026-04-22", "FEV1/FVC: 0.68 — mild obstructive pattern"                  ),
        (5, 3, "Echocardiogram"               , "PENDING"  , "2026-05-01", None        , None                                                         ),
        (6, 3, "BNP Level"                    , "COMPLETED", "2026-05-01", "2026-05-02", "BNP: 450 pg/mL (elevated; reference <100 pg/mL)"            ),
        (7, 5, "HbA1c"                        , "COMPLETED", "2026-04-28", "2026-04-30", "HbA1c: 8.2% (above target of <7%)"                          ),
        (8, 5, "Microalbumin/Creatinine Ratio", "PENDING"  , "2026-05-05", None        , None                                                         ),
    ])

    c.executemany("INSERT OR REPLACE INTO appointments VALUES (?,?,?,?,?,?)", [
        (1, 1, "Endocrinology Follow-up" , "2026-05-15", "SCHEDULED", "Review HbA1c and diabetes management plan"   ),
        (2, 2, "Pulmonology Review"      , "2026-05-12", "SCHEDULED", "Asthma control and inhaler technique"        ),
        (3, 3, "Cardiology Urgent Review", "2026-05-10", "SCHEDULED", "Review elevated BNP; adjust diuretic therapy"),
        (4, 4, "Psychiatry Follow-up"    , "2026-05-20", "SCHEDULED", "Antidepressant response evaluation"          ),
        (5, 5, "Endocrinology Follow-up" , "2026-05-18", "SCHEDULED", "Diabetes management — HbA1c 8.2%"            ),
    ])

    conn.commit()
    conn.close()

create_patient_database()

conn = sqlite3.connect(DB_PATH)
for t in ["patients", "exams", "appointments"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"   {t}: {n} rows")
conn.close()
print(f"✅ Patient DB ready: {DB_PATH}")

   patients: 5 rows
   exams: 8 rows
   appointments: 5 rows
✅ Patient DB ready: /Users/anabarbosa/fiap/fiap-tech-challenge/module_03/data/patients.db


## 4 - RAG with FAISS (Medical Knowledge Base)

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
import os

# Disable OpenMP parallelization to avoid threading crash in local runs
os.environ["OMP_NUM_THREADS"] = "1"

MEDQUAD_PATH = str(DATA_DIR / "processed" / "medical_data.jsonl")

def load_and_chunk_medical_docs(path: str, chunk_size: int = 900, chunk_overlap: int = 150) -> list[Document]:
    """Load medical Q&A from JSONL with source preservation and chunk them for better retrieval."""
    docs = []
    with open(path) as f:
        for line in f:
            item = json.loads(line)
            text = item["text"]
            source = item.get("source", "Unknown")
            
            if "Question:" not in text:
                continue
            
            docs.append(Document(page_content=text, metadata={"source": source}))
    
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunked = []
    for doc in docs:
        chunks = splitter.split_text(doc.page_content)
        for chunk in chunks:
            chunked.append(Document(page_content=chunk, metadata=doc.metadata))
    return chunked

print(f"Loading and chunking medical documents…")
medical_docs = load_and_chunk_medical_docs(MEDQUAD_PATH)
print(f"Loaded {len(medical_docs)} chunks")

# all-MiniLM-L6-v2 is small (~80 MB) and runs fine on CPU
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)

print("Building FAISS index…")
vectorstore = FAISS.from_documents(medical_docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"✅ FAISS index ready — {len(medical_docs)} chunks")

Loading and chunking medical documents…
Loaded 15204 chunks


/var/folders/jg/dxyd05n13qb115cb0brztg6c0000gn/T/ipykernel_83604/2452164662.py:39: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
2026-05-24 17:54:06,101 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-24 17:54:06,103 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-24 17:54:06,140 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199b

Building FAISS index…


2026-05-24 17:58:02,592 [INFO] Loading faiss.
2026-05-24 17:58:02,659 [INFO] Successfully loaded faiss.


✅ FAISS index ready — 15204 chunks


## 5 - LangGraph Pipeline

In [6]:
from langgraph.graph import StateGraph, END

# ── State ─────────────────────────────────────────────────────────────────────

class AssistantState(TypedDict):
    patient_id:        Optional[int]
    query:             str
    patient_context:   str
    pending_exams:     List[dict]
    recent_results:    List[dict]
    docs:              List[dict]
    medical_knowledge: str
    sources:           List[str]
    response:          str
    is_safe:           bool
    log_entry:         dict

# ── DB helper ─────────────────────────────────────────────────────────────────

def fetch_patient_data(patient_id: int) -> dict:
    conn = sqlite3.connect(DB_PATH)
    c    = conn.cursor()
    patient   = c.execute("SELECT * FROM patients WHERE id=?", (patient_id,)).fetchone()
    pending   = c.execute(
        "SELECT exam_type, ordered_date FROM exams WHERE patient_id=? AND status='PENDING'",
        (patient_id,)).fetchall()
    completed = c.execute(
        """SELECT exam_type, result_date, results FROM exams
           WHERE patient_id=? AND status='COMPLETED' AND results IS NOT NULL
           ORDER BY result_date DESC LIMIT 3""",
        (patient_id,)).fetchall()
    next_appt = c.execute(
        """SELECT appointment_type, scheduled_date, notes FROM appointments
           WHERE patient_id=? AND status='SCHEDULED' ORDER BY scheduled_date LIMIT 1""",
        (patient_id,)).fetchone()
    conn.close()
    return {"patient": patient, "pending": pending, "completed": completed, "next_appt": next_appt}

# ── Node 1: get_patient_context ───────────────────────────────────────────────

def get_patient_context(state: AssistantState) -> AssistantState:
    pid = state.get("patient_id")
    if not pid:
        return {**state, "patient_context": "", "pending_exams": [], "recent_results": []}

    d = fetch_patient_data(pid)
    p = d["patient"]
    if not p:
        return {**state, "patient_context": "Patient not found.", "pending_exams": [], "recent_results": []}

    lines = [
        f"Patient: {p[1]}, {p[2]}y, {p[3]}",
        f"Diagnosis: {p[4]}",
        f"Medications: {p[5]}",
        f"Allergies: {p[6]}",
        f"Blood type: {p[7]}",
        f"Physician: {p[8]}  |  Last visit: {p[9]}",
    ]
    if d["next_appt"]:
        a = d["next_appt"]
        lines.append(f"Next appointment: {a[0]} on {a[1]} — {a[2]}")
    if d["completed"]:
        lines.append("Recent results:")
        for exam, date, results in d["completed"]:
            lines.append(f"  • {exam} ({date}): {results}")

    return {
        **state,
        "patient_context": "\n".join(lines),
        "pending_exams":   [{"exam": r[0], "ordered": r[1]} for r in d["pending"]],
        "recent_results":  [{"exam": r[0], "date": r[1], "results": r[2]} for r in d["completed"]],
    }

# ── Node 2: retrieve_medical_knowledge ───────────────────────────────────────

def retrieve_medical_knowledge(state: AssistantState) -> AssistantState:
    query = state["query"]
    # enrich with patient diagnosis so retrieval is more relevant
    for line in state.get("patient_context", "").splitlines():
        if line.startswith("Diagnosis:"):
            query = f"{query} {line}"
            break

    docs_retrieved = retriever.invoke(query)
    docs_slim = [{"text": d.page_content, "source": d.metadata.get("source", "")} for d in docs_retrieved]
    
    knowledge = "\n\n".join(
        f"[Source {i+1} | {d['source']}]: {d['text'][:300]}"
        for i, d in enumerate(docs_slim)
    )
    sources = [f"{d['source']}" for d in docs_slim]
    return {**state, "docs": docs_slim, "medical_knowledge": knowledge, "sources": sources}

# ── Node 3: generate_response ─────────────────────────────────────────────────

def generate_response(state: AssistantState) -> AssistantState:
    parts = []
    if state["patient_context"]:
        parts.append(f"PATIENT INFORMATION:\n{state['patient_context']}")
    if state["pending_exams"]:
        exam_list = "\n".join(f"  - {e['exam']} (ordered {e['ordered']})" for e in state["pending_exams"])
        parts.append(f"PENDING EXAMS (results not yet available):\n{exam_list}")
    if state["medical_knowledge"]:
        parts.append(f"RELEVANT MEDICAL KNOWLEDGE:\n{state['medical_knowledge']}")
    parts.append(f"DOCTOR'S QUESTION:\n{state['query']}")
    parts.append(
        "INSTRUCTIONS: Give evidence-based clinical guidance. "
        "Never make a direct prescription decision — always flag that treatment changes require physician sign-off. "
        "If exams are pending, recommend awaiting results before changing therapy."
    )
    response = llm.invoke("\n\n".join(parts))
    return {**state, "response": response}

print("✅ Nodes 1-3 defined")

✅ Nodes 1-3 defined


In [7]:
# ── Node 4: validate_response (safety guardrails) ────────────────────────────

UNSAFE_PATTERNS = [
    "i prescribe", "prescribe this", "take this medication",
    "start taking", "increase the dose to", "stop all medications",
    "you should take", "i recommend taking", "administer",
]
DISCLAIMER = (
    "\n\n---\n"
    "⚠️  Human Validation Required — This is informational support only. "
    "All prescriptions, dosage changes, and treatment decisions must be reviewed "
    "and approved by the attending physician before implementation."
)

def validate_response(state: AssistantState) -> AssistantState:
    response = state["response"]
    is_safe  = not any(p in response.lower() for p in UNSAFE_PATTERNS)

    if not is_safe:
        response = (
            "⚠️  SAFETY FLAG: Direct prescription language detected — physician review mandatory.\n\n"
            + response
        )

    response += DISCLAIMER

    if state.get("sources"):
        response += "\n\n📚 Sources used:"
        for src in state["sources"]:
            response += f"\n- {src}"

    return {**state, "response": response, "is_safe": is_safe}

# ── Node 5: log_interaction (audit trail) ────────────────────────────────────

AUDIT_LOG = str(OUTPUTS_DIR / "audit_log.jsonl")

def log_interaction(state: AssistantState) -> AssistantState:
    entry = {
        "timestamp":           datetime.now().isoformat(),
        "patient_id":          state.get("patient_id"),
        "query":               state["query"],
        "sources_used":        state.get("sources", []),
        "pending_exams_found": len(state.get("pending_exams", [])),
        "is_safe":             state.get("is_safe", True),
        "response_chars":      len(state.get("response", "")),
    }
    with open(AUDIT_LOG, "a") as f:
        f.write(json.dumps(entry) + "\n")

    flag = "" if entry["is_safe"] else " [SAFETY FLAG]"
    logger.info(
        f"patient={entry['patient_id']} | query='{entry['query'][:60]}' | "
        f"sources={len(entry['sources_used'])} | pending={entry['pending_exams_found']}{flag}"
    )
    return {**state, "log_entry": entry}

# ── Compile graph ─────────────────────────────────────────────────────────────

workflow = StateGraph(AssistantState)
workflow.add_node("get_patient_context",        get_patient_context)
workflow.add_node("retrieve_medical_knowledge", retrieve_medical_knowledge)
workflow.add_node("generate_response",          generate_response)
workflow.add_node("validate_response",          validate_response)
workflow.add_node("log_interaction",            log_interaction)

workflow.set_entry_point("get_patient_context")
workflow.add_edge("get_patient_context",        "retrieve_medical_knowledge")
workflow.add_edge("retrieve_medical_knowledge", "generate_response")
workflow.add_edge("generate_response",          "validate_response")
workflow.add_edge("validate_response",          "log_interaction")
workflow.add_edge("log_interaction",             END)

assistant = workflow.compile()

print("✅ LangGraph pipeline compiled")
print("   Nodes:", list(workflow.nodes.keys()))
print(assistant.get_graph().draw_ascii())

✅ LangGraph pipeline compiled
   Nodes: ['get_patient_context', 'retrieve_medical_knowledge', 'generate_response', 'validate_response', 'log_interaction']
        +-----------+          
        | __start__ |          
        +-----------+          
               *               
               *               
               *               
    +---------------------+    
    | get_patient_context |    
    +---------------------+    
               *               
               *               
               *               
+----------------------------+ 
| retrieve_medical_knowledge | 
+----------------------------+ 
               *               
               *               
               *               
    +-------------------+      
    | generate_response |      
    +-------------------+      
               *               
               *               
               *               
    +-------------------+      
    | validate_response |      
    +--------

## 6 - Demo: Contextualized Clinical Queries

In [8]:
def show_chunks(docs: list[dict], max_chars: int = 220):
    """Display retrieved chunks with source and preview."""
    for i, d in enumerate(docs, 1):
        preview = d["text"].replace("\n", " ")[:max_chars]
        print(f"Chunk {i} | Source: {d['source']}")
        print(f"Preview: {preview}...\n")

def ask(query: str, patient_id: Optional[int] = None, verbose: bool = True) -> dict:
    """Run the medical assistant pipeline on a query."""
    result = assistant.invoke({
        "patient_id":        patient_id,
        "query":             query,
        "patient_context":   "",
        "pending_exams":     [],
        "recent_results":    [],
        "docs":              [],
        "medical_knowledge": "",
        "sources":           [],
        "response":          "",
        "is_safe":           True,
        "log_entry":         {},
    })
    if verbose:
        sep = "=" * 70
        print(f"\n{sep}\nQUERY: {query}")
        if patient_id:
            print(f"PATIENT ID: {patient_id}")
        print(sep)
        if result["patient_context"]:
            print(f"\n📋 PATIENT CONTEXT:\n{result['patient_context']}")
        if result["pending_exams"]:
            print(f"\n⚠️  PENDING EXAMS: {[e['exam'] for e in result['pending_exams']]}")
        print(f"\n🤖 RESPONSE:\n{result['response']}")
        docs_used = result.get("docs", [])
        if docs_used:
            print(f"\n📚 Chunks retrieved ({len(docs_used)}):\n")
            show_chunks(docs_used)
        print(f"\n🕐 {result['log_entry'].get('timestamp','N/A')}  |  safe={result['is_safe']}")
    return result

In [9]:
# Carlos Oliveira (id=3) — CHF + AFib, BNP 450, pending Echocardiogram
r1 = ask(
    "What does an elevated BNP of 450 pg/mL indicate in a heart failure patient and what should we monitor?",
    patient_id=3,
)

2026-05-24 17:58:32,091 [INFO] patient=3 | query='What does an elevated BNP of 450 pg/mL indicate in a heart f' | sources=3 | pending=1



QUERY: What does an elevated BNP of 450 pg/mL indicate in a heart failure patient and what should we monitor?
PATIENT ID: 3

📋 PATIENT CONTEXT:
Patient: Carlos Oliveira, 72y, M
Diagnosis: Chronic Heart Failure, Atrial Fibrillation
Medications: Furosemide 40mg, Warfarin 5mg, Bisoprolol 5mg
Allergies: Sulfa drugs
Blood type: B+
Physician: Dr. Ana Costa  |  Last visit: 2026-05-01
Next appointment: Cardiology Urgent Review on 2026-05-10 — Review elevated BNP; adjust diuretic therapy
Recent results:
  • BNP Level (2026-05-02): BNP: 450 pg/mL (elevated; reference <100 pg/mL)

⚠️  PENDING EXAMS: ['Echocardiogram']

🤖 RESPONSE:
Based on the information provided and drawing from established medical knowledge sources such as MedQuAD and The Human Phenotype Ontology (HPO), an elevated BNP level of 450 pg/mL in a patient with chronic heart failure, like Carlos Oliveira, is indicative of worsening cardiac function. Brain Natriuretic Peptide (BNP) levels are commonly used to assess the severity and

In [10]:
# Roberto Ferreira (id=5) — T2DM, HbA1c 8.2%, pending microalbumin ratio
r2 = ask(
    "Patient's HbA1c is 8.2%, above the 7% target. What are the evidence-based next steps for intensifying diabetes management?",
    patient_id=5,
)

2026-05-24 17:58:58,910 [INFO] patient=5 | query='Patient's HbA1c is 8.2%, above the 7% target. What are the e' | sources=3 | pending=1



QUERY: Patient's HbA1c is 8.2%, above the 7% target. What are the evidence-based next steps for intensifying diabetes management?
PATIENT ID: 5

📋 PATIENT CONTEXT:
Patient: Roberto Ferreira, 55y, M
Diagnosis: Type 2 Diabetes, Dyslipidemia
Medications: Metformin 1000mg, Atorvastatin 20mg
Allergies: Aspirin
Blood type: A-
Physician: Dr. Pedro Lima  |  Last visit: 2026-04-28
Next appointment: Endocrinology Follow-up on 2026-05-18 — Diabetes management — HbA1c 8.2%
Recent results:
  • HbA1c (2026-04-30): HbA1c: 8.2% (above target of <7%)

⚠️  PENDING EXAMS: ['Microalbumin/Creatinine Ratio']

🤖 RESPONSE:
Given the patient Roberto Ferreira's current HbA1c level of 8.2%, which is above the target range for individuals with type 2 diabetes (<7%), several evidence-based steps can be considered to intensify his diabetes management, pending physician review and consideration:

1. **Review Current Medications**: Assess if Roberto's current medication regimen is optimal or requires adjustment base

In [11]:
# João Silva (id=1) — Penicillin allergy, allergy check for Amoxicillin
r3 = ask(
    "Can this patient receive Amoxicillin for a bacterial infection?",
    patient_id=1,
)

2026-05-24 17:59:26,512 [INFO] patient=1 | query='Can this patient receive Amoxicillin for a bacterial infecti' | sources=3 | pending=2 [SAFETY FLAG]



QUERY: Can this patient receive Amoxicillin for a bacterial infection?
PATIENT ID: 1

📋 PATIENT CONTEXT:
Patient: João Silva, 65y, M
Diagnosis: Hypertension, Type 2 Diabetes
Medications: Metformin 500mg, Lisinopril 10mg
Allergies: Penicillin
Blood type: A+
Physician: Dr. Ana Costa  |  Last visit: 2026-04-15
Next appointment: Endocrinology Follow-up on 2026-05-15 — Review HbA1c and diabetes management plan
Recent results:
  • Lipid Panel (2026-04-18): Total Cholesterol: 210 mg/dL, LDL: 130 mg/dL, HDL: 45 mg/dL

⚠️  PENDING EXAMS: ['HbA1c', 'Renal Function Panel']

🤖 RESPONSE:
⚠️  SAFETY FLAG: Direct prescription language detected — physician review mandatory.

Based on the information provided and following best practices for patient care management, I would advise caution in administering Amoxicillin to João Silva due to his allergy profile which includes a noted penicillin allergy. Penicillin allergies can sometimes be associated with other beta-lactam antibiotics like amoxicillin; t

In [12]:
# General query — no patient context
r4 = ask("What are the current treatment guidelines for asthma in adults?")

2026-05-24 17:59:53,650 [INFO] patient=None | query='What are the current treatment guidelines for asthma in adul' | sources=3 | pending=0



QUERY: What are the current treatment guidelines for asthma in adults?

🤖 RESPONSE:
Current asthma management in adults is guided by the National Asthma Education and Prevention Program (NAEPP) guidelines which emphasize a stepwise approach to treatment based on individual patient's needs assessed through regular monitoring of symptoms, lung function tests like spirometry or peak flow measurements, and response to therapy.

Initially, all adult asthma patients should be prescribed inhaled corticosteroids (ICS) as the mainstay treatment unless they have a history of exacerbations that are not well controlled with ICS alone. In such cases or if there is evidence of fixed airflow obstruction on spirometry, additional medications like long-acting beta2 agonists (LABAs) and/or leukotriene receptor antagonists may be added to the regimen as per Step 3 guidelines [Source: NAEPP].

For patients with persistent asthma who are not well controlled on high doses of ICS, LABA or combination therap

## 7 - Automated Flow: Daily Pending Exam Alert

In [13]:
def run_pending_exam_alert_flow():
    """Automated workflow: generate a clinical briefing for every patient with pending exams."""
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute("""
        SELECT DISTINCT p.id, p.name, p.diagnosis, p.doctor,
               GROUP_CONCAT(e.exam_type, ', ') AS pending
        FROM patients p
        JOIN exams e ON e.patient_id = p.id
        WHERE e.status = 'PENDING'
        GROUP BY p.id
    """).fetchall()
    conn.close()

    print("=" * 70)
    print("🏥  AUTOMATED PENDING EXAM ALERT — DAILY BRIEFING")
    print(f"    {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print("=" * 70)

    report = []
    for pid, name, diagnosis, doctor, pending in rows:
        print(f"\n👤 {name} (ID {pid}) — {doctor}")
        print(f"   Diagnosis: {diagnosis}")
        print(f"   Pending:   {pending}")

        result = ask(
            f"Patient has pending exams: {pending}. Given their diagnosis ({diagnosis}), "
            f"what clinical actions should be prioritised while awaiting these results?",
            patient_id=pid,
            verbose=False,
        )
        excerpt = result["response"][:300].replace("\n", " ")
        print(f"   Recommendation: {excerpt}…")

        report.append({
            "patient_id":    pid,
            "patient_name":  name,
            "doctor":        doctor,
            "pending_exams": pending,
            "recommendation": result["response"][:600],
        })

    ts = datetime.now().strftime("%Y%m%d_%H%M")
    report_path = OUTPUTS_DIR / f"alert_report_{ts}.json"
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False))
    print(f"\n✅ Report saved: {report_path}")

run_pending_exam_alert_flow()

🏥  AUTOMATED PENDING EXAM ALERT — DAILY BRIEFING
    2026-05-24 17:59

👤 João Silva (ID 1) — Dr. Ana Costa
   Diagnosis: Hypertension, Type 2 Diabetes
   Pending:   HbA1c, Renal Function Panel


2026-05-24 18:00:22,143 [INFO] patient=1 | query='Patient has pending exams: HbA1c, Renal Function Panel. Give' | sources=3 | pending=2


   Recommendation: Given the patient's diagnosis of hypertension and type 2 diabetes, it is crucial to maintain a comprehensive approach while awaiting further test results such as HbA1c and Renal Function Panel outcomes. Here are evidence-based clinical guidelines for managing these conditions:  **Hypertension Manage…

👤 Carlos Oliveira (ID 3) — Dr. Ana Costa
   Diagnosis: Chronic Heart Failure, Atrial Fibrillation
   Pending:   Echocardiogram


2026-05-24 18:00:48,857 [INFO] patient=3 | query='Patient has pending exams: Echocardiogram. Given their diagn' | sources=3 | pending=1


   Recommendation: Given the patient Carlos Oliveira's diagnosis of Chronic Heart Failure (CHF) and Atrial Fibrillation (AF), along with his current medications—furosemide for fluid management, warfarin as an anticoagulant due to AF-related stroke risk, and bisoprolol for rate control in AF—and considering the elevate…

👤 Roberto Ferreira (ID 5) — Dr. Pedro Lima
   Diagnosis: Type 2 Diabetes, Dyslipidemia
   Pending:   Microalbumin/Creatinine Ratio


2026-05-24 18:01:15,470 [INFO] patient=5 | query='Patient has pending exams: Microalbumin/Creatinine Ratio. Gi' | sources=3 | pending=1


   Recommendation: Given the patient Roberto Ferreira's diagnosis of Type 2 Diabetes and Dyslipidemia, along with his current medication regimen (Metformin 1000mg for diabetes management and Atorvastatin 20mg for dyslipidemia), it is important to approach any changes in therapy cautiously until we have the results of …

✅ Report saved: /Users/anabarbosa/fiap/fiap-tech-challenge/module_03/outputs/alert_report_20260524_1801.json


## 8 - Audit Log Viewer

In [15]:
import pandas as pd

def view_audit_log():
    if not Path(AUDIT_LOG).exists():
        print("No audit log yet.")
        return

    entries = [json.loads(l) for l in Path(AUDIT_LOG).read_text().splitlines() if l.strip()]
    df = pd.DataFrame(entries)
    df["timestamp"]     = pd.to_datetime(df["timestamp"])
    df["sources_count"] = df["sources_used"].apply(len)
    df = df.drop(columns=["sources_used"])

    print(f"Audit log — {len(df)} interactions\n")
    cols = ["timestamp", "patient_id", "query", "pending_exams_found", "sources_count", "is_safe", "response_chars"]
    print(df[cols].to_string(index=False))

    unsafe = df[~df["is_safe"]]
    if not unsafe.empty:
        print(f"\n⚠️  {len(unsafe)} interaction(s) flagged:")
        print(unsafe[["timestamp", "patient_id", "query"]].to_string(index=False))
    else:
        print("\n✅ All interactions passed safety validation.")

view_audit_log()

Audit log — 7 interactions

                 timestamp  patient_id                                                                                                                                                                                      query  pending_exams_found  sources_count  is_safe  response_chars
2026-05-24 17:58:32.090467         3.0                                                                                     What does an elevated BNP of 450 pg/mL indicate in a heart failure patient and what should we monitor?                    1              3     True            2364
2026-05-24 17:58:58.909224         5.0                                                                 Patient's HbA1c is 8.2%, above the 7% target. What are the evidence-based next steps for intensifying diabetes management?                    1              3     True            2317
2026-05-24 17:59:26.511255         1.0                                                                         

## Done!